# Phase 2: Data Cleaning and Financial Feature Engineering
**Objective:** Clean the Superstore dataset, handle anomalies, standardize data definitions, and engineer key financial metrics for Profit & Loss (P&L) analysis according to standard accounting principles.

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Load raw dataset
df = pd.read_csv('../data/raw/superstore_raw.csv')
print(f"Initial Shape: {df.shape}")

## Part 1: Data Cleaning

In [ ]:
# 1. Standardize column names (lowercase with underscore)
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
print("Standardized Columns:", df.columns.tolist())

In [ ]:
# 2. Handle missing values
missing_check = df.isnull().sum()
print("Missing values before cleaning:\n", missing_check[missing_check > 0])

# Superstore dataset typically has missing 'postal_code' for some cities.
# Since postal code isn't critical for P&L analysis, we will impute with a placeholder '00000'.
if 'postal_code' in df.columns:
    df['postal_code'] = df['postal_code'].fillna(0)
    df['postal_code'] = df['postal_code'].astype(int).astype(str).str.zfill(5)

In [ ]:
# 3. Remove duplicates
duplicates = df.duplicated().sum()
if duplicates > 0:
    print(f"Found {duplicates} duplicates, removing...")
    df = df.drop_duplicates()
else:
    print("No duplicates found.")

In [ ]:
# 4. Fix data types
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# Validate logical consistency: Ship Date must be >= Order Date
invalid_dates = df[df['ship_date'] < df['order_date']]
if not invalid_dates.empty:
    print(f"Found {len(invalid_dates)} records with ship_date before order_date. Dropping them...")
    df = df[df['ship_date'] >= df['order_date']]

In [ ]:
# 5. Handle Obvious Outliers/Errors in Financial metrics
# Sales & Quantity should be strictly positive. Discount between 0 and 1.
df = df[df['sales'] > 0]
df = df[df['quantity'] > 0]
df = df[(df['discount'] >= 0) & (df['discount'] <= 1)]

print(f"Shape after cleaning: {df.shape}")

## Part 2: Financial Feature Engineering
**Note on accounting:** In the Superstore dataset structure, 'Profit' represents Gross Profit (Contribution Margin). Consequently, COGS is implicitly extracted by treating Sales - Profit = COGS.

In [ ]:
# 1. Standard Accounting Metrics
df['cogs'] = df['sales'] - df['profit']
df['gross_profit'] = df['sales'] - df['cogs'] # Aligns with the raw 'profit' mathematically.

df['gross_margin_pct'] = (df['gross_profit'] / df['sales']) * 100
df['profit_margin_pct'] = (df['profit'] / df['sales']) * 100

In [ ]:
# 2. Discount Impact Metrics
df['discount_amount'] = df['sales'] * df['discount']
df['discount_impact_pct'] = (df['discount_amount'] / df['sales']) * 100

In [ ]:
# 3. Time-based attributes
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['quarter'] = df['order_date'].dt.quarter
df['month_year'] = df['order_date'].dt.to_period('M').astype(str)

In [ ]:
# 4. Unit Economics
df['profit_per_unit'] = df['profit'] / df['quantity']

In [ ]:
# 5. Dimension Profit Ratios 
# Calculating the ratio of row profit against overall category profit directly via transformation.
category_group = df.groupby('category')['profit'].transform('sum')
df['category_profit_ratio'] = (df['profit'] / category_group) * 100

In [ ]:
# 6. YoY Growth Calculation
# Row-level YoY is not logically coherent. We form a separate monthly aggregated time-series matrix.
monthly_matrix = df.groupby('month_year')[['sales', 'profit']].sum().reset_index()
monthly_matrix['month_year'] = pd.to_datetime(monthly_matrix['month_year'])
monthly_matrix = monthly_matrix.sort_values('month_year')

monthly_matrix['yoy_sales_growth_pct'] = monthly_matrix['sales'].pct_change(periods=12) * 100
monthly_matrix['yoy_profit_growth_pct'] = monthly_matrix['profit'].pct_change(periods=12) * 100

print("--- Monthly Time Series Aggregation with YoY (Last 5 periods) ---")
display(monthly_matrix.tail())

## Part 3: Final Steps

In [ ]:
# Display Summary of newly engineered P&L features
new_metrics = ['cogs', 'gross_margin_pct', 'profit_margin_pct', 'discount_amount', 'discount_impact_pct', 'profit_per_unit']
print("\n--- Summary of New P&L Features ---")
display(df[new_metrics].describe().round(2))

In [ ]:
# Save the datasets to the processed directory
cleaned_path = '../data/processed/superstore_cleaned.csv'
df.to_csv(cleaned_path, index=False)
print(f"\nCore Cleaned dataset successfully saved to: {cleaned_path}")

monthly_matrix_path = '../data/processed/monthly_yoy_matrix.csv'
monthly_matrix.to_csv(monthly_matrix_path, index=False)
print(f"YoY Aggregation matrix successfully saved to: {monthly_matrix_path}")